# Experimental Flood Analysis System

Hampton Roads Flood Analysis, Old Dominion University, Summer 2026.

Team:

- Andrew Mounsey, amounsey@odu.edu, https://andrew3784.github.io/
- Ryan Thompson, rthom035@odu.com, https://gameguyalien.github.io
- Scott Zumwalt, jzum001@odu.edu, https://drainsmith.github.io

Project goal: build a Python/PostGIS flood-analysis workflow that uses public water-level, elevation, road, and building-footprint data to estimate screening-level flood exposure for Norfolk, Virginia.


## How To Run This Notebook

This is a hybrid notebook: it is readable as a project walkthrough, but it can also rerun the workflow when the local environment is ready.

Run order:

1. Run the setup and validation cells first.
2. Review each section before running long-running cells.
3. Keep `RUN_LONG_STEPS = False` for a presentation/demo or when outputs already exist.
4. Set `RUN_LONG_STEPS = True` only when you want to download data and regenerate outputs.

Requirements:

- Python environment with this project installed, usually `python -m pip install -e .`.
- PostgreSQL/PostGIS database reachable through `.env` variable `DATABASE_URL`.
- Enough disk space for the 1-meter DEM workflow. The DEM source tiles are about 2.36 GB before outputs.
- Internet access for NOAA, USGS, TIGER, and Overture download steps.


## Data Sources

Primary workflow data sources used by the current implementation:

- NOAA Tides and Currents station `8638610`, Sewells Point, VA, for observed water levels and station datum offsets.
- USGS 3DEP 1-meter DEM, `VA_HamptonRoads_B23`, for elevation.
- US Census TIGER/Line roads for a public road-centerline baseline.
- Overture Maps release `2026-06-17.0` for building footprints.

Early project notes also considered NOAA Sea Level Rise Viewer DEM files and Hampton Roads elevation-certificate building footprints. Those remain useful context, but the active reproducible workflow uses the sources listed above.


## Modeling Scope And Limitations

The model is a screening-level static water-surface analysis. It compares scenario water-surface elevations to DEM elevations, then applies a boundary-connectivity filter to remove isolated inland depressions.

It is not a hydrodynamic model. It does not simulate flow timing, culverts, barriers, drainage systems, wave effects, rainfall, or local hydraulic controls. Results are appropriate for workflow validation and preliminary exposure screening, not final engineering decisions.


## 1. Environment Setup

Run this section first. It defines shared settings and helper functions used by later cells.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
RUN_LONG_STEPS = False

STUDY_AREA_ID = "norfolk_va"
STUDY_AREA_NAME = "Norfolk, VA"
NORFOLK_GEOID = "51710"
NOAA_STATION = "8638610"
EVENT_BEGIN_DATE = "20260627"
EVENT_END_DATE = "20260628"
EVENT_START = "2026-06-27T00:00:00Z"
EVENT_END = "2026-06-28T23:59:59Z"

DEM_1M = Path("data/processed/dem/norfolk_va_usgs_1m_hamptonroads_b23_navd88_m.tif")
DEPTH_DIR_1M = Path("data/processed/flood_depths_1m")
CONNECTED_DEPTH_DIR_1M = Path("data/processed/flood_depths_connected_1m")
BUILDINGS_PARQUET = Path("data/raw/overture/norfolk_buildings.parquet")
GIS_OUTPUT = Path("data/processed/gis/norfolk_flood_exposure_1m.gpkg")

load_dotenv()
print(f"Python: {sys.version.split()[0]}")
print(f"Project root: {PROJECT_ROOT}")
print(f"DATABASE_URL configured: {bool(os.environ.get('DATABASE_URL'))}")
print(f"RUN_LONG_STEPS: {RUN_LONG_STEPS}")


In [ ]:
def run_command(args, *, long_running=False, check=True):
    """Run a project command and print useful notebook output."""
    command_text = " ".join(str(part) for part in args)
    if long_running and not RUN_LONG_STEPS:
        print(f"Skipped long-running step: {command_text}")
        return None

    print(f"Running: {command_text}")
    completed = subprocess.run(args, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {command_text}")
    return completed


## 2. Database And PostGIS Checks

These cells verify that the database exists, PostGIS is available, and project schemas/tables can be created. If these fail, check `.env`, database credentials, and PostGIS privileges before continuing.


In [ ]:
run_command([sys.executable, "scripts/check_postgis.py"])


In [ ]:
run_command([sys.executable, "scripts/init_db.py"])


## 3. NOAA Water Levels And Flood Scenarios

This section loads observed water levels for Sewells Point, creates peak-water scenarios, and converts scenario elevations from MLLW to NAVD88 so they can be compared with the DEM.


In [ ]:
run_command([
    sys.executable, "scripts/ingest_noaa_water_levels.py",
    "--station", NOAA_STATION,
    "--begin-date", EVENT_BEGIN_DATE,
    "--end-date", EVENT_END_DATE,
    "--datum", "MLLW",
    "--units", "english",
    "--interval", "6",
], long_running=True)


In [ ]:
run_command([
    sys.executable, "scripts/create_peak_scenarios.py",
    "--event-start", EVENT_START,
    "--event-end", EVENT_END,
    "--study-area", "Norfolk pilot",
])

run_command([sys.executable, "scripts/ingest_noaa_datums.py", "--station", NOAA_STATION], long_running=True)

run_command([
    sys.executable, "scripts/convert_scenarios_datum.py",
    "--station", NOAA_STATION,
    "--source-datum", "MLLW",
    "--target-datum", "NAVD88",
])


## 4. Study Area And 1-Meter DEM

The Norfolk study area is loaded from TIGER/Line county-equivalent boundaries. The active elevation source is USGS 3DEP 1-meter `VA_HamptonRoads_B23`. Downloading and clipping the DEM are long-running steps, so they are guarded by `RUN_LONG_STEPS`.


In [ ]:
run_command([
    sys.executable, "scripts/ingest_study_area.py",
    "--study-area-id", STUDY_AREA_ID,
    "--geoid", NORFOLK_GEOID,
    "--name", STUDY_AREA_NAME,
], long_running=True)


In [ ]:
run_command([
    sys.executable, "scripts/download_usgs_dem.py",
    "--study-area-id", STUDY_AREA_ID,
    "--dataset", "Digital Elevation Model (DEM) 1 meter",
    "--title-contains", "VA_HamptonRoads_B23",
    "--max-results", "100",
])

run_command([
    sys.executable, "scripts/download_usgs_dem.py",
    "--study-area-id", STUDY_AREA_ID,
    "--dataset", "Digital Elevation Model (DEM) 1 meter",
    "--title-contains", "VA_HamptonRoads_B23",
    "--max-results", "100",
    "--output-dir", "data/raw/usgs_3dep_1m",
    "--workers", "4",
    "--download",
], long_running=True)


In [ ]:
dem_tiles = sorted(Path("data/raw/usgs_3dep_1m").glob("*.tif"))
if dem_tiles:
    run_command([
        sys.executable, "scripts/clip_dem_to_study_area.py",
        "--study-area-id", STUDY_AREA_ID,
        "--input-dem", *map(str, dem_tiles),
        "--output", str(DEM_1M),
    ], long_running=True)
else:
    print("No DEM tiles found under data/raw/usgs_3dep_1m. Run the download step first.")


In [ ]:
paths_to_check = [DEM_1M, DEPTH_DIR_1M, CONNECTED_DEPTH_DIR_1M, BUILDINGS_PARQUET, GIS_OUTPUT]
pd.DataFrame(
    {"path": [str(path) for path in paths_to_check], "exists": [path.exists() for path in paths_to_check]}
)


## 5. Flood-Depth Rasters And Connected Extents

Flood-depth rasters are created by subtracting DEM elevation from each scenario water-surface elevation. The connected-depth step removes wet cells that are not connected to the modeled boundary water source. The polygonization step creates PostGIS vector extents for exposure overlays and GIS display.


In [ ]:
run_command([
    sys.executable, "scripts/create_flood_depth_rasters.py",
    "--study-area-id", STUDY_AREA_ID,
    "--dem", str(DEM_1M),
    "--dem-units", "meters",
    "--output-dir", str(DEPTH_DIR_1M),
], long_running=True)

run_command([
    sys.executable, "scripts/create_connected_flood_depth_rasters.py",
    "--output-dir", str(CONNECTED_DEPTH_DIR_1M),
], long_running=True)

run_command([sys.executable, "scripts/polygonize_connected_flood_extents.py", "--min-depth-ft", "0"], long_running=True)


## 6. Road And Building Exposure

Road exposure uses TIGER road centerlines. Building exposure uses Overture building footprints clipped to Norfolk. Building maximum depths are sampled from connected 1-meter rasters, so polygon-only sliver contacts without wet raster pixels are excluded.


In [ ]:
run_command([
    sys.executable, "scripts/ingest_roads.py",
    "--study-area-id", STUDY_AREA_ID,
    "--geoid", NORFOLK_GEOID,
], long_running=True)

run_command([sys.executable, "scripts/calculate_road_exposure.py", "--study-area-id", STUDY_AREA_ID, "--connected"])


In [ ]:
run_command([
    sys.executable, "scripts/download_overture_buildings.py",
    "--study-area-id", STUDY_AREA_ID,
    "--output", str(BUILDINGS_PARQUET),
], long_running=True)

run_command([
    sys.executable, "scripts/ingest_buildings_from_file.py",
    "--study-area-id", STUDY_AREA_ID,
    "--input", str(BUILDINGS_PARQUET),
    "--id-column", "id",
    "--name-column", "name",
    "--type-column", "building_type",
    "--source", "Overture Maps 2026-06-17.0",
], long_running=True)

run_command([sys.executable, "scripts/calculate_building_exposure.py", "--study-area-id", STUDY_AREA_ID], long_running=True)


## 7. GIS Export

This exports the study area layers, connected extents, roads, buildings, and exposure outputs to a GeoPackage for ArcGIS or QGIS. Adjacent CSV files contain scenario summaries.


In [ ]:
run_command([
    sys.executable, "scripts/export_gis_layers.py",
    "--study-area-id", STUDY_AREA_ID,
    "--output", str(GIS_OUTPUT),
], long_running=True)


## 8. Database Result Summaries

The cells below query PostGIS for compact result summaries. They are useful for confirming that the workflow ran successfully and for quickly presenting project findings.


In [ ]:
from sqlalchemy import create_engine, text

def read_query(sql, params=None):
    database_url = os.environ.get("DATABASE_URL")
    if not database_url:
        raise RuntimeError("DATABASE_URL is not configured.")
    engine = create_engine(database_url)
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn, params=params or {})


In [ ]:
read_query("""
SELECT 'processed.flood_scenarios' AS table_name, count(*) AS row_count FROM processed.flood_scenarios
UNION ALL SELECT 'processed.connected_flood_extents', count(*) FROM processed.connected_flood_extents
UNION ALL SELECT 'processed.roads', count(*) FROM processed.roads
UNION ALL SELECT 'processed.buildings', count(*) FROM processed.buildings
UNION ALL SELECT 'results.connected_road_exposure_summary', count(*) FROM results.connected_road_exposure_summary
UNION ALL SELECT 'results.connected_building_exposure_summary', count(*) FROM results.connected_building_exposure_summary
ORDER BY table_name
""")


In [ ]:
read_query("""
SELECT scenario_id, road_count, flooded_road_count, round(flooded_length_mi::numeric, 3) AS flooded_length_mi
FROM results.connected_road_exposure_summary
WHERE study_area_id = :study_area_id
ORDER BY scenario_id
""", {"study_area_id": STUDY_AREA_ID})


In [ ]:
read_query("""
SELECT scenario_id, building_count, flooded_building_count, round(flooded_footprint_area_m2::numeric, 1) AS flooded_footprint_area_m2
FROM results.connected_building_exposure_summary
WHERE study_area_id = :study_area_id
ORDER BY scenario_id
""", {"study_area_id": STUDY_AREA_ID})


## Current Validated Results

The latest validated 1-meter workflow produced these project-level summaries:

- 1-meter clipped DEM: EPSG:26918, 1 x 1 meter resolution, 24,284 x 19,010 cells, 176,426,700 valid pixels.
- Roads: 6,663 TIGER road features.
- Buildings: 83,891 Overture building footprints clipped to Norfolk.
- Connected road exposure: 12.439, 13.724, 19.770, and 43.972 flooded road miles for current, +1 ft, +2 ft, and +3 ft scenarios.
- Connected building exposure: 169, 196, 512, and 1,773 flooded buildings for current, +1 ft, +2 ft, and +3 ft scenarios.

The exported GeoPackage path is `data/processed/gis/norfolk_flood_exposure_1m.gpkg`.


## Troubleshooting

Common problems and fixes:

- `DATABASE_URL configured: False`: create or update `.env` with a valid PostGIS connection string.
- PostGIS check fails: verify the database has the PostGIS extension and that the connected user has the required schema/table privileges.
- DEM clip says no tiles found: run the USGS download cell with `RUN_LONG_STEPS = True`, or place GeoTIFFs under `data/raw/usgs_3dep_1m/`.
- Building download is slow: Overture extraction is a bulk data step and requires internet access plus `duckdb` and `pyarrow`.
- Empty exposure summaries: confirm scenarios, connected extents, roads/buildings, and connected raster paths exist before running exposure calculations.
- ArcGIS/QGIS cannot open outputs: rerun the export step and verify `data/processed/gis/norfolk_flood_exposure_1m.gpkg` exists.
